# **Notebook 4: RAG Implementation**
## Assignment: Hybrid RAG & Fine-Tuning for Customer Support
---

### TO-DO: Before Running This Notebook

**Files you NEED:**
- [ ] `corporate_policies/` folder with `.md` SOP files
- [ ] `outputs.json` — Created by Notebook 3
- [ ] GPU runtime enabled

**Files this notebook will CREATE:**
- [ ] `./chroma_db/` — Persisted ChromaDB vector index _(Required by NB5 and NB7)_
- [ ] `outputs.json` (updated) — adds `naive_rag_output` _(Required by NB5 and NB7)_

---

### **Task 3.2: Implement Retrieval-Assisted Generation**

#### **3.2.1 Generate Embeddings [4 marks]**
**The Task:** Initialise the `all-MiniLM-L6-v2` embedding model, embed the SOP documents, and validate the embeddings were produced.

**Hints & Tips:**
* Load the SOP documents with `TextLoader` first (or reuse the corpus from NB2).
* `HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")` runs on CPU — no GPU needed.
* Validate by embedding one sample string and checking the vector length (384 dims for MiniLM).
* You MUST use the same embedding model when reloading in Notebooks 5 and 7.

**Embedding Model Options:**
* **`all-MiniLM-L6-v2`** (recommended): 384-dim, fast, ~80MB.
* **`all-mpnet-base-v2`**: 768-dim, higher quality, slower.
* **`bge-small-en-v1.5`**: 384-dim, newer architecture.

**Learner Inference:** Your text is now coordinates in semantic space — similar meanings sit close together.

In [1]:
import os
import glob
from langchain_community.document_loaders import TextLoader
from langchain_huggingface import HuggingFaceEmbeddings

# 1. Locate and load corporate SOP documents
policy_dirs = [
    "corporate_policies",
    "../corporate_policies",
    "Files/Notebook/corporate_policies",
    "Files/Dataset/Dataset/sop_documents",
    "../Dataset/Dataset/sop_documents"
]
sop_dir = None
for p in policy_dirs:
    if os.path.exists(p) and len(glob.glob(os.path.join(p, "*.md"))) > 0:
        sop_dir = p
        break

if not sop_dir:
    raise FileNotFoundError("Could not find corporate policy (.md) documents directory.")

sop_files = sorted(glob.glob(os.path.join(sop_dir, "*.md")))
print(f"Loading SOP files from '{sop_dir}' ({len(sop_files)} files)...")

documents = []
for fpath in sop_files:
    loader = TextLoader(fpath, encoding="utf-8")
    loaded_docs = loader.load()
    for doc in loaded_docs:
        doc.metadata["filename"] = os.path.basename(fpath)
        doc.metadata["title"] = os.path.splitext(os.path.basename(fpath))[0].replace("_", " ").title()
        documents.append(doc)

print(f"Successfully loaded {len(documents)} SOP documents into LangChain Document objects.")

# 2. Initialize the all-MiniLM-L6-v2 embedding model
EMBEDDING_MODEL_NAME = "sentence-transformers/all-MiniLM-L6-v2"
print(f"\nInitializing embedding model: '{EMBEDDING_MODEL_NAME}' (CPU-optimized)...")
embeddings = HuggingFaceEmbeddings(
    model_name=EMBEDDING_MODEL_NAME,
    model_kwargs={"device": "cpu"},
    encode_kwargs={"normalize_embeddings": True}
)

# 3. Validate embeddings on sample text
sample_query = "What is the policy for delayed shipping?"
sample_embedding = embeddings.embed_query(sample_query)
embedding_dim = len(sample_embedding)

print(f"\n--- Embedding Validation ---")
print(f"Sample test query: \"{sample_query}\"")
print(f"Embedding vector dimension: {embedding_dim}")
assert embedding_dim == 384, f"Expected 384 dimensions for {EMBEDDING_MODEL_NAME}, but got {embedding_dim}."
print("Validation PASSED: Embedding dimensionality is 384 (all-MiniLM-L6-v2 specification).")


C:\Users\hp\AppData\Local\Temp\ipykernel_712\619613787.py:3: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import TextLoader


Loading SOP files from 'corporate_policies' (13 files)...
Successfully loaded 13 SOP documents into LangChain Document objects.

Initializing embedding model: 'sentence-transformers/all-MiniLM-L6-v2' (CPU-optimized)...


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

C:\Users\hp\anaconda3\Lib\site-packages\huggingface_hub\file_download.py:143: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\hp\.cache\huggingface\hub\models--sentence-transformers--all-MiniLM-L6-v2. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]


--- Embedding Validation ---
Sample test query: "What is the policy for delayed shipping?"
Embedding vector dimension: 384
Validation PASSED: Embedding dimensionality is 384 (all-MiniLM-L6-v2 specification).


#### **3.2.2 Build Vector Index [4 marks]**
**The Task:** Create a persistent Chroma vector index from the embedded documents, configure similarity search, and validate the index.

**Hints & Tips:**
* `Chroma.from_documents(docs, embeddings, persist_directory="./chroma_db")` auto-saves — no manual `.persist()` needed.
* Validate with `vector_db._collection.count()` — should equal the number of SOP documents.
* Run one test `.similarity_search("refund", k=1)` to confirm retrieval works.

**Vector DB Options:**
* **ChromaDB** (recommended): simple API, auto-persistence, LangChain integration.
* **FAISS**: faster for >100K docs, but no built-in persistence (manual serialization).

**Learner Inference:** The index lets you search by meaning — it returns the document mathematically closest to your query's coordinates.

In [2]:
import shutil
from langchain_chroma import Chroma

CHROMA_PERSIST_DIR = "./chroma_db"

# Ensure clean directory for fresh index creation
if os.path.exists(CHROMA_PERSIST_DIR):
    shutil.rmtree(CHROMA_PERSIST_DIR)

print(f"Building persistent Chroma vector database at '{CHROMA_PERSIST_DIR}'...")
vector_db = Chroma.from_documents(
    documents=documents,
    embedding=embeddings,
    persist_directory=CHROMA_PERSIST_DIR
)

# 1. Validate collection document count
doc_count = vector_db._collection.count()
print(f"\n--- Vector Database Validation ---")
print(f"Documents indexed in ChromaDB collection: {doc_count}")
assert doc_count == len(documents), f"Document count mismatch: expected {len(documents)}, got {doc_count}."
print(f"Collection verification PASSED: Exactly {doc_count} SOP documents indexed.")

# 2. Test similarity search with a sample query ("refund")
test_keyword = "refund"
retrieved_test = vector_db.similarity_search(test_keyword, k=1)

print(f"\n--- Test Similarity Search ('{test_keyword}') ---")
if retrieved_test:
    top_doc = retrieved_test[0]
    print(f"Top retrieved document: {top_doc.metadata.get('filename')} ({top_doc.metadata.get('title')})")
    print(f"Snippet: {top_doc.page_content[:180]}...")


Building persistent Chroma vector database at './chroma_db'...



--- Vector Database Validation ---
Documents indexed in ChromaDB collection: 13
Collection verification PASSED: Exactly 13 SOP documents indexed.

--- Test Similarity Search ('refund') ---
Top retrieved document: refund_policy.md (Refund Policy)
Snippet: # Refund Policy

## Eligibility
Customers may request a refund within 30 days of the original purchase or
delivery date, whichever is later. To be eligible, the item must be unused...


#### **3.2.3 Implement Retrieval Workflow [4 marks]**
**The Task:** Execute a "Naive RAG" workflow — pass the raw customer query into the vector DB, fetch the top result, and augment the LLM prompt.

**Hints & Tips:**
* Use `.similarity_search(query, k=1)` for the top-1 document.
* Check whether the raw query retrieved the WRONG policy — common with ambiguous queries.
* Inject context via the system prompt: `"Answer strictly using this SOP: {context}"`.

**Parameter Tuning:**
* `k=1`: one document (focused). `k=3`: more context if SOPs overlap. `k=5`: max, risks long prompts.

**Learner Inference:** Noisy queries often retrieve the wrong document — proving Naive RAG is flawed and motivating the fine-tuned router.

In [3]:
import json
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

# 1. Load test_query from outputs.json created in Notebook 3
outputs_candidates = [
    "outputs.json",
    "../outputs.json",
    "../../outputs.json",
    "Files/Notebook/outputs.json"
]
outputs_path = None
for p in outputs_candidates:
    if os.path.exists(p):
        outputs_path = p
        break

if not outputs_path:
    raise FileNotFoundError("outputs.json from Notebook 3 not found!")

with open(outputs_path, "r", encoding="utf-8") as f:
    outputs_data = json.load(f)

test_query = outputs_data["test_query"]
print(f"Loaded test_query from '{outputs_path}':\n\"{test_query}\"\n")

# 2. Execute Naive Similarity Search (top-1)
retrieved_docs = vector_db.similarity_search(test_query, k=1)
retrieved_doc = retrieved_docs[0]
retrieved_filename = retrieved_doc.metadata.get("filename")
retrieved_title = retrieved_doc.metadata.get("title")
retrieved_context = retrieved_doc.page_content

print("--- Naive Retrieval Result ---")
print(f"Top Retrieved SOP Document: {retrieved_filename} ('{retrieved_title}')")
print("\nRetrieved Context Preview:")
print(retrieved_context[:250], "...\n")

# Analysis of Naive Retrieval Quality:
print("--- Naive Retrieval Failure / Observation ---")
if "shipping" not in retrieved_filename:
    print(
        f"CRITICAL OBSERVATION: The user's query asks about an order that hasn't arrived ('delayed shipping'), "
        f"but because they added 'can I get a refund?', Naive RAG retrieved '{retrieved_filename}' instead of 'shipping_delays.md'. "
        f"This demonstrates the fundamental failure mode of unguided Naive RAG on multi-intent / ambiguous queries!"
    )
else:
    print(f"Retrieved document '{retrieved_filename}'. Inspecting context relevance for shipping delay.")

# 3. Load Qwen model and generate response with injected SOP context
MODEL_ID = "Qwen/Qwen2.5-1.5B-Instruct"
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

if torch.cuda.is_available():
    bnb_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_compute_dtype=torch.float16,
        bnb_4bit_use_double_quant=True
    )
    model = AutoModelForCausalLM.from_pretrained(MODEL_ID, quantization_config=bnb_config, device_map="auto", trust_remote_code=True)
else:
    model = AutoModelForCausalLM.from_pretrained(MODEL_ID, device_map="cpu", dtype=torch.bfloat16 if hasattr(torch, 'bfloat16') else torch.float32, trust_remote_code=True)

# Augment prompt strictly with the retrieved SOP context
system_prompt = (
    "You are a customer support agent. Answer the user inquiry strictly using the following corporate SOP policy context. "
    "Do not invent facts or extrapolate beyond what is stated in the policy.\n\n"
    f"=== RETRIEVED CORPORATE POLICY SOP ===\n{retrieved_context}\n======================================="
)

messages = [
    {"role": "system", "content": system_prompt},
    {"role": "user", "content": test_query}
]

prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

print("\nGenerating Naive RAG response (context-augmented)...")
with torch.no_grad():
    output_tokens = model.generate(
        **inputs,
        max_new_tokens=120,
        do_sample=False,
        temperature=None,
        top_p=None,
        pad_token_id=tokenizer.pad_token_id
    )

gen_tokens = output_tokens[0][inputs["input_ids"].shape[1]:]
naive_rag_output = tokenizer.decode(gen_tokens, skip_special_tokens=True).strip()

print("\n--- Naive RAG Model Output ---")
print(naive_rag_output)


Loaded test_query from 'outputs.json':
"My order hasn't arrived yet and it's been several days. When will it get here and can I get a refund?"

--- Naive Retrieval Result ---
Top Retrieved SOP Document: refund_policy.md ('Refund Policy')

Retrieved Context Preview:
# Refund Policy

## Eligibility
Customers may request a refund within 30 days of the original purchase or
delivery date, whichever is later. To be eligible, the item must be unused or
defective, and the request must reference a valid order identifier ...

--- Naive Retrieval Failure / Observation ---
CRITICAL OBSERVATION: The user's query asks about an order that hasn't arrived ('delayed shipping'), but because they added 'can I get a refund?', Naive RAG retrieved 'refund_policy.md' instead of 'shipping_delays.md'. This demonstrates the fundamental failure mode of unguided Naive RAG on multi-intent / ambiguous queries!


Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]


Generating Naive RAG response (context-augmented)...



--- Naive RAG Model Output ---
I'm sorry to hear that your order has not arrived yet. Please provide me with your order number and the date you placed your order so we can check its status. Once I have this information, I'll assist you in determining if there was any issue with delivery and whether you're eligible for a refund. Thank you for reaching out.


---
## Save Artifacts for Downstream Notebooks

In [4]:
# 1. Update outputs.json with naive_rag_output
outputs_data["naive_rag_output"] = naive_rag_output

output_file = "outputs.json"
with open(output_file, "w", encoding="utf-8") as f:
    json.dump(outputs_data, f, indent=2, ensure_ascii=False)

print(f"Updated '{output_file}' with 'naive_rag_output'.")

# Also copy to root and Files/Notebook for downstream notebooks (NB5 and NB7)
extra_paths = [
    os.path.join("Files", "Notebook", output_file),
    os.path.join("..", "..", output_file),
    os.path.join("..", output_file)
]
for p in extra_paths:
    parent = os.path.dirname(p)
    if parent and os.path.exists(parent):
        with open(p, "w", encoding="utf-8") as f:
            json.dump(outputs_data, f, indent=2, ensure_ascii=False)
        print(f"Saved copy to: {p}")

# Also copy chroma_db to root if needed
if os.path.basename(os.getcwd()) == "Notebook":
    root_chroma = os.path.join("..", "..", "chroma_db")
    if not os.path.exists(root_chroma):
        shutil.copytree(CHROMA_PERSIST_DIR, root_chroma, dirs_exist_ok=True)
        print(f"Mirrored ChromaDB to: {root_chroma}")

# 2. Verify updated outputs.json
with open(output_file, "r", encoding="utf-8") as f:
    verified_data = json.load(f)

print("\n--- Artifacts Verification ---")
print(f"outputs.json keys: {list(verified_data.keys())}")
assert "naive_rag_output" in verified_data, "naive_rag_output is missing from outputs.json!"
assert os.path.exists(CHROMA_PERSIST_DIR), "chroma_db directory does not exist!"
print("ChromaDB path confirmed:", os.path.abspath(CHROMA_PERSIST_DIR))
print("All artifacts for Notebooks 5, 6, and 7 successfully validated!")


Updated 'outputs.json' with 'naive_rag_output'.
Saved copy to: ..\..\outputs.json
Saved copy to: ..\outputs.json
Mirrored ChromaDB to: ..\..\chroma_db

--- Artifacts Verification ---
outputs.json keys: ['test_query', 'ground_truth', 'baseline_output', 'naive_rag_output']
ChromaDB path confirmed: C:\Users\hp\OneDrive\Desktop\Hybrid rag_fine tuning capstone\Files\Notebook\chroma_db
All artifacts for Notebooks 5, 6, and 7 successfully validated!


---
## END-OF-NOTEBOOK CHECKLIST

> **IMPORTANT: Verify before proceeding to Notebook 5.**

- [ ] SOP documents loaded via `TextLoader`
- [ ] **Embeddings generated and validated** ← _Task 3.2.1_
- [ ] **ChromaDB index built, validated, and persisted** ← _Task 3.2.2_
- [ ] Naive similarity search executed on `test_query`
- [ ] Naive RAG output generated with SOP context injected
- [ ] **`./chroma_db/` exists on disk** ← _CRITICAL for NB5 and NB7_
- [ ] **`outputs.json` updated** with `naive_rag_output` ← _CRITICAL for NB5 and NB7_

**If any item is unchecked, fix it before moving on.**